# RSNA Knee Abnormality Detection: End-to-End GPU Training Pipeline
---

## Architecture and Pipeline Overview
This self-contained notebook trains the custom **Multi-View 2.5D Knee Study Model** with **12-Head Finding-Specific Attention Pooling** directly on Kaggle GPUs (T4 x2 / P100).

### Core Components:
1. **Dynamic Annotation Discovery:** Automatically merges competition `train.csv` with LLM pseudo-labels or falls back to the 58 gold cases.
2. **Stratified 5-Fold Partitioning:** Preserves human gold-standard (58 cases) and pseudo-label ratios.
3. **Spatial Standardization:** Bilinear interpolation standardizes variable MRI matrix resolutions (960x960, 800x800) to 384x384.
4. **Plane Conditioning:** Embeds Sagittal (0), Coronal (1), and Axial (2) plane semantics.
5. **Finding-Specific Attention Pooling:** Dedicated attention weights for each of the 12 pathologies.
6. **Finding-Weighted BCE Loss:** Rebalances rare pathologies (MCL, Lateral OA) against frequent findings (Effusion).
7. **Weight Export:** Writes `best_model.pt` to `/kaggle/working/best_model.pt` for direct submission.

In [ ]:
import gc
import os
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import torchvision.models as tv_models

try:
    import pydicom
except ImportError:
    pydicom = None

try:
    import timm
except ImportError:
    timm = None

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")

## 1. Environment Configuration and Hyperparameters

In [ ]:
class Config:
    TARGET_COLS = [
        'ACL',
        'MCL',
        'Medial Meniscus',
        'Lateral Meniscus',
        'Medial OA',
        'Lateral OA',
        'PF OA',
        'Effusion',
        'Synovitis',
        "Baker's",
        'Contusion',
        'Fracture'
    ]
    NUM_CLASSES = len(TARGET_COLS)  # 12
    
    # Training hyperparameters
    BACKBONE = 'resnet34'
    PRETRAINED = True
    IMAGE_SIZE = 384
    MAX_WINDOWS_PER_STUDY = 48
    BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 2  # Effective batch size: 4
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    EPOCHS = 5
    NUM_FOLDS = 5
    ACTIVE_FOLD = 0
    NUM_WORKERS = 2
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../experiments')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_competition_path() -> Path:
    """Finds the directory containing train.csv and train_series.csv."""
    if Path('/kaggle/input').exists():
        for p in Path('/kaggle/input').rglob('train.csv'):
            if (p.parent / 'train_series.csv').exists():
                return p.parent
        p_std = Path('/kaggle/input/rsna-knee-abnormality-detection')
        if p_std.exists():
            return p_std
    for loc in [Path('data'), Path('../data'), Path('../../data'), Path('.')]:
        if (loc / 'train.csv').exists():
            return loc.resolve()
    return Path('data').resolve()

Config.DATA_DIR = resolve_competition_path()
print(f"Resolved DATA_DIR: {Config.DATA_DIR}")
print(f"Target Output Dir: {Config.OUTPUT_DIR}")
print(f"Target Device    : {Config.DEVICE}")

## 2. Dynamic Training Cohort Ingestion

In [ ]:
def load_training_cohort(data_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Loads or builds the training dataset:
    1. If train_annotated.csv is present, loads all 4,407 cases.
    2. If llm_labels*.csv is attached, merges with train.csv on the fly.
    3. If neither is found, trains directly on the 58 gold-standard cases.
    """
    train_csv = data_dir / 'train.csv'
    series_csv = data_dir / 'train_series.csv'
    
    if not train_csv.exists() or not series_csv.exists():
        raise FileNotFoundError(f"Missing competition train files in {data_dir}")
        
    df_raw = pd.read_csv(train_csv)
    df_series = pd.read_csv(series_csv)
    
    # 1. Search for pre-distilled train_annotated.csv
    for candidate in [data_dir / 'train_annotated.csv', Path('data/train_annotated.csv'), Path('../data/train_annotated.csv')]:
        if candidate.exists():
            print(f"Loaded pre-distilled cohort: {candidate}")
            df = pd.read_csv(candidate)
            df['is_gold'] = df_raw.set_index('StudyInstanceUID').loc[df['StudyInstanceUID'], 'ACL'].notna().values
            return df, df_series
            
    # 2. Search for LLM pseudo-labels in /kaggle/input
    llm_files = list(Path('/kaggle/input').rglob('llm_labels*.csv')) if Path('/kaggle/input').exists() else []
    if len(llm_files) > 0:
        llm_path = sorted(llm_files)[-1]
        print(f"Distilling cohort dynamically from LLM pseudo-labels: {llm_path}")
        llm_df = pd.read_csv(llm_path).set_index('StudyInstanceUID')
        
        df_merged = df_raw.copy()
        gold_mask = df_merged['ACL'].notna()
        df_merged['is_gold'] = gold_mask
        
        for idx, row in df_merged[~gold_mask].iterrows():
            uid = row['StudyInstanceUID']
            if uid in llm_df.index:
                for col in Config.TARGET_COLS:
                    if col in llm_df.columns:
                        df_merged.at[idx, col] = float(llm_df.loc[uid, col])
                        
        # Fill any remaining NaNs with column means
        for col in Config.TARGET_COLS:
            df_merged[col] = df_merged[col].fillna(df_merged[col].mean())
            
        print(f"Dynamically created annotated cohort: {len(df_merged)} studies ({gold_mask.sum()} gold, {(~gold_mask).sum()} pseudo-labeled)")
        return df_merged, df_series
        
    # 3. Fallback: Train strictly on the 58 gold-standard human cases
    gold_df = df_raw[df_raw['ACL'].notna()].copy().reset_index(drop=True)
    gold_df['is_gold'] = True
    print(f"NOTICE: No pseudo-labels dataset detected.")
    print(f"Training on {len(gold_df)} gold-standard human annotated cases.")
    print(f"[Tip] To train on all 4,407 studies, attach dataset: stevenleehans/rsna-knee-llm-report-labels via '+ Add Input'")
    return gold_df, df_series

df_all, df_series = load_training_cohort(Config.DATA_DIR)
print(f"Total Training Cohort: {len(df_all)} studies across {len(df_series)} series")

## 3. DICOM Preprocessing and Multiview Windowing

In [ ]:
PLANE_MAP = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}

def load_dicom_slice(dcm_path: Union[str, Path], target_size: int = Config.IMAGE_SIZE) -> np.ndarray:
    """Reads DICOM, calibrates intensity, applies 1st-99th percentile normalization, and standardizes spatial shape."""
    if pydicom is None:
        return np.zeros((target_size, target_size), dtype=np.float32)
    
    dcm = pydicom.dcmread(str(dcm_path))
    arr = dcm.pixel_array.astype(np.float32)
    
    slope = getattr(dcm, 'RescaleSlope', 1.0)
    intercept = getattr(dcm, 'RescaleIntercept', 0.0)
    if slope != 1.0 or intercept != 0.0:
        arr = arr * slope + intercept
        
    if getattr(dcm, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        arr = np.amax(arr) - arr
        
    p_low, p_high = np.percentile(arr, (1, 99))
    if p_high > p_low:
        arr = np.clip(arr, p_low, p_high)
        arr = (arr - p_low) / (p_high - p_low)
    else:
        arr = np.zeros_like(arr)
        
    # Standardize spatial dimensions (resolves 960x960, 800x800 variable resolutions)
    if arr.shape != (target_size, target_size):
        t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(target_size, target_size), mode='bilinear', align_corners=False)
        arr = t.squeeze().numpy()
        
    return arr.astype(np.float32)

def create_25d_windows(slices: List[np.ndarray], target_size: int = Config.IMAGE_SIZE) -> np.ndarray:
    """Stacks consecutive 2D slices into 3-channel 2.5D RGB windows."""
    if len(slices) < 3:
        base = slices[0] if len(slices) > 0 else np.zeros((target_size, target_size), dtype=np.float32)
        triplet = np.stack([base, base, base], axis=0)
        return triplet[np.newaxis, ...]
    
    windows = []
    for i in range(1, len(slices) - 1):
        triplet = np.stack([slices[i - 1], slices[i], slices[i + 1]], axis=0)
        windows.append(triplet)
    return np.array(windows, dtype=np.float32)

## 4. Dataset, Stratification, and DataLoader Pipeline

In [ ]:
class KneeStudyDataset(Dataset):
    """Study-level PyTorch Dataset standardizing MRI series into 2.5D window representations."""
    def __init__(self, df: pd.DataFrame, series_df: pd.DataFrame, data_dir: Path, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.data_dir = data_dir
        self.is_test = is_test
        self.series_by_study = {uid: grp for uid, grp in series_df.groupby('StudyInstanceUID')}
        
    def __len__(self) -> int:
        return len(self.df)
        
    def __getitem__(self, idx: int) -> Dict[str, Union[torch.Tensor, str]]:
        study_id = str(self.df.iloc[idx]['StudyInstanceUID'])
        study_series = self.series_by_study.get(study_id, None)
        
        all_windows = []
        plane_codes = []
        
        if study_series is not None:
            for _, srow in study_series.iterrows():
                series_id = str(srow['SeriesInstanceUID'])
                plane = srow.get('Anatomical_Plane', 'Sagittal')
                plane_code = PLANE_MAP.get(plane, 0)
                
                candidates = [
                    self.data_dir / 'train_series' / study_id / series_id,
                    self.data_dir / 'train_series' / series_id,
                    self.data_dir / 'test_series' / study_id / series_id,
                    self.data_dir / study_id / series_id
                ]
                series_folder = next((c for c in candidates if c.exists()), None)
                
                slices = []
                if series_folder is not None:
                    dcm_files = sorted(list(series_folder.glob('*.dcm')))
                    n_files = len(dcm_files)
                    s_idx = int(n_files * 0.10)
                    e_idx = max(s_idx + 3, int(n_files * 0.90))
                    
                    for fpath in dcm_files[s_idx:e_idx]:
                        try:
                            slices.append(load_dicom_slice(fpath))
                        except Exception:
                            continue
                            
                if len(slices) >= 3:
                    w = create_25d_windows(slices)
                    all_windows.append(w)
                    plane_codes.extend([plane_code] * len(w))
                    
        if len(all_windows) == 0:
            # Synthetic placeholder for missing or unmounted volumes
            placeholder = torch.zeros((16, 3, Config.IMAGE_SIZE, Config.IMAGE_SIZE), dtype=torch.float32)
            planes = torch.tensor([0, 1, 2] * 6, dtype=torch.long)[:16]
            final_windows = placeholder
            final_planes = planes
        else:
            concatenated = np.concatenate(all_windows, axis=0)
            if len(concatenated) > Config.MAX_WINDOWS_PER_STUDY:
                sub_indices = np.linspace(0, len(concat_windows if 'concat_windows' in locals() else concatenated) - 1, Config.MAX_WINDOWS_PER_STUDY, dtype=int)
                concatenated = concatenated[sub_indices]
                plane_codes = [plane_codes[i] for i in sub_indices]
                
            final_windows = torch.tensor(concatenated, dtype=torch.float32)
            final_planes = torch.tensor(plane_codes, dtype=torch.long)
            
        item = {
            'study_id': study_id,
            'images': final_windows,
            'planes': final_planes
        }
        
        if not self.is_test:
            targets = self.df.iloc[idx][Config.TARGET_COLS].values.astype(np.float32)
            item['targets'] = torch.tensor(targets, dtype=torch.float32)
            
        return item

def collate_studies(batch):
    study_ids = [b['study_id'] for b in batch]
    images = [b['images'] for b in batch]
    planes = [b['planes'] for b in batch]
    res = {'study_ids': study_ids, 'images': images, 'planes': planes}
    if 'targets' in batch[0]:
        res['targets'] = torch.stack([b['targets'] for b in batch], dim=0)
    return res

## 5. Multi-View Architecture: 12-Head Finding-Specific Attention

In [ ]:
class FindingSpecificAttentionPooling(nn.Module):
    """12-Head Attention Pooling: Allocates dedicated attention weights per pathology."""
    def __init__(self, in_features: int, num_classes: int = Config.NUM_CLASSES, hidden_dim: int = 128):
        super().__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.attention_v = nn.Linear(in_features, hidden_dim)
        self.attention_w = nn.Linear(hidden_dim, num_classes)
        self.classifiers = nn.ModuleList([nn.Linear(in_features, 1) for _ in range(num_classes)])
        
    def forward(self, features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        att_scores = self.attention_w(torch.tanh(self.attention_v(features)))
        att_weights = F.softmax(att_scores.transpose(0, 1), dim=-1)
        context = torch.matmul(att_weights, features)
        logits = torch.stack([
            self.classifiers[k](context[k]).squeeze(-1) for k in range(self.num_classes)
        ], dim=-1)
        return logits, att_weights

class KneeStudyModel(nn.Module):
    """Study-Level Vision Model with Plane Conditioning and 12-Head Attention."""
    def __init__(self, backbone_name: str = Config.BACKBONE, pretrained: bool = Config.PRETRAINED, num_classes: int = Config.NUM_CLASSES):
        super().__init__()
        if timm is not None:
            try:
                self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
            except Exception as e:
                print(f"Internet unavailable ({e}). Fallback to offline backbone initialization.")
                self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0)
            in_features = self.backbone.num_features
        else:
            res = tv_models.resnet34(weights=tv_models.ResNet34_Weights.DEFAULT if pretrained else None)
            in_features = res.fc.in_features
            res.fc = nn.Identity()
            self.backbone = res
            
        self.plane_emb = nn.Embedding(3, in_features)
        self.pool = FindingSpecificAttentionPooling(in_features=in_features, num_classes=num_classes)
        
    def forward(self, study_images: List[torch.Tensor], study_planes: List[torch.Tensor]) -> Dict[str, torch.Tensor]:
        batch_logits = []
        for img, pl in zip(study_images, study_planes):
            feat = self.backbone(img)
            feat = feat + self.plane_emb(pl)
            logits, _ = self.pool(feat)
            batch_logits.append(logits)
        return {'logits': torch.stack(batch_logits, dim=0)}

## 6. Training, Evaluation, and Checkpoint Export

In [ ]:
# Stratified 5-Fold split (preserving gold cases ratio)
np.random.seed(42)
df_all['fold'] = -1
gold_mask = df_all['is_gold']

gold_idx = df_all[gold_mask].index.to_numpy().copy()
np.random.shuffle(gold_idx)
for i, idx in enumerate(gold_idx):
    df_all.loc[idx, 'fold'] = i % Config.NUM_FOLDS
    
pseudo_idx = df_all[~gold_mask].index.to_numpy().copy()
if len(pseudo_idx) > 0:
    np.random.shuffle(pseudo_idx)
    for i, idx in enumerate(pseudo_idx):
        df_all.loc[idx, 'fold'] = i % Config.NUM_FOLDS

train_df = df_all[df_all['fold'] != Config.ACTIVE_FOLD].reset_index(drop=True)
val_df = df_all[df_all['fold'] == Config.ACTIVE_FOLD].reset_index(drop=True)

print(f"Fold {Config.ACTIVE_FOLD}: {len(train_df)} Train Studies | {len(val_df)} Val Studies")

# DataLoaders
train_loader = DataLoader(
    KneeStudyDataset(train_df, df_series, Config.DATA_DIR),
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_studies,
    num_workers=Config.NUM_WORKERS
)
val_loader = DataLoader(
    KneeStudyDataset(val_df, df_series, Config.DATA_DIR),
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_studies,
    num_workers=Config.NUM_WORKERS
)

# Model, Loss, Optimizer
device = torch.device(Config.DEVICE)
model = KneeStudyModel().to(device)

# Positive weights for class rebalancing
pos_weights = []
for col in Config.TARGET_COLS:
    pos = (train_df[col] > 0.5).sum()
    ratio = max(1.0, min((len(train_df) - pos) / max(1, pos), 4.0))
    pos_weights.append(ratio)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, dtype=torch.float32, device=device))

optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)
scaler = GradScaler() if device.type == 'cuda' else None

# Training Execution Loop
best_auc = 0.0
best_ckpt = Config.OUTPUT_DIR / 'best_model.pt'

print(f"\nStarting Training on {device}...")
for epoch in range(1, Config.EPOCHS + 1):
    t0 = time.time()
    model.train()
    total_train_loss = 0.0
    optimizer.zero_grad()
    
    for step, batch in enumerate(train_loader):
        imgs = [x.to(device) for x in batch['images']]
        pls = [x.to(device) for x in batch['planes']]
        targets = batch['targets'].to(device)
        
        use_amp = scaler is not None and device.type == 'cuda'
        with autocast(enabled=use_amp):
            out = model(imgs, pls)
            loss = criterion(out['logits'], targets) / Config.GRAD_ACCUM_STEPS
            
        if use_amp:
            scaler.scale(loss).backward()
            if (step + 1) % Config.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            loss.backward()
            if (step + 1) % Config.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                optimizer.step()
                optimizer.zero_grad()
                
        total_train_loss += loss.item() * Config.GRAD_ACCUM_STEPS
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation Loop
    model.eval()
    val_preds, val_targets = [], []
    total_val_loss = 0.0
    
    with torch.no_grad():
        for batch in val_loader:
            imgs = [x.to(device) for x in batch['images']]
            pls = [x.to(device) for x in batch['planes']]
            targets = batch['targets'].to(device)
            
            out = model(imgs, pls)
            loss = criterion(out['logits'], targets)
            total_val_loss += loss.item()
            
            val_preds.append(torch.sigmoid(out['logits']).cpu().numpy())
            val_targets.append(targets.cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader)
    y_p = np.concatenate(val_preds, axis=0)
    y_t = np.concatenate(val_targets, axis=0)
    
    # Compute Macro ROC-AUC
    aucs = []
    for c in range(Config.NUM_CLASSES):
        if len(np.unique(y_t[:, c])) > 1:
            try:
                aucs.append(float(roc_auc_score(y_t[:, c], y_p[:, c])))
            except Exception:
                pass
    val_auc = float(np.mean(aucs)) if len(aucs) > 0 else 0.50
    
    scheduler.step()
    dur = time.time() - t0
    
    is_best = val_auc > best_auc
    if is_best:
        best_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'val_auc': val_auc,
            'backbone': Config.BACKBONE,
            'target_cols': Config.TARGET_COLS
        }, best_ckpt)
        
    print(f"Epoch {epoch:02d}/{Config.EPOCHS:02d} [{dur:.1f}s] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro AUC: {val_auc:.4f} {'(BEST)' if is_best else ''}")

print("\n=======================================================")
print(f"  TRAINING COMPLETED: Best Macro AUC = {best_auc:.4f}")
print(f"  Model Weights Exported to: {best_ckpt.resolve()}")
print("=======================================================")